# Comparação de Modelos e Análise de Trade-off (Etapa 2)

Este _notebook_ busca os experimentos mais recentes no `MLFlow` (Regressão Logística, Dummy e nosso **MLP criado em PyTorch**) e apresenta a [Tabela Comparativa] necessária para os entregáveis. Por fim, gera a análise de **Trade-off de Custos (Falso Positivo vs Falso Negativo)**.

In [ ]:
import os
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Setup MLFlow Tracking URI
root_dir = os.path.dirname(os.getcwd())
mlflow.set_tracking_uri(f"sqlite:///{os.path.join(root_dir, 'mlflow.db')}")


### Tabela Comparativa via MLFlow
O código abaixo varre todos os experimentos e captura a versão que perfomou melhor no conjunto de testes de cada modelo para comparar Lado-a-Lado.

In [ ]:
# 2. Busca e construção da Tabela Comparativa
experiments = [exp.experiment_id for exp in mlflow.search_experiments()]
runs = mlflow.search_runs(experiment_ids=experiments)

# Filtra e renomeia colunas para uma visualização clara
cols_of_interest = ['tags.stage', 'params.model_type', 'metrics.test_accuracy', 'metrics.test_f1_score', 'metrics.test_precision', 'metrics.test_recall', 'metrics.test_roc_auc']

if not runs.empty:
    available_cols = [c for c in cols_of_interest if c in runs.columns]
    comparative_table = runs[available_cols].copy()
    comparative_table.columns = [c.split('.')[-1].replace('_', ' ').title() for c in comparative_table.columns]
    # Ordena pelo F1-Score do teste (descendente)
    comparative_table = comparative_table.sort_values(by='Test F1 Score', ascending=False).reset_index(drop=True)
    display(comparative_table)
else:
    print("Nenhum run encontrado no MLFlow. Certifique-se de executar `python src/train.py`.")

### Análise de Custo (Trade-off: Falsos Positivos vs Falsos Negativos)
Num cenário de *Churn*, reter um cliente prestes a cancelar (previnir um **Falso Negativo**) costuma economizar o CLTV futuro do cliente.
Oferecer descontos a quem não ia cancelar (**Falso Positivo**) gera um custo da verba promocional desperdiçada.


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import torch

# Exemplo de simulação de carga de dados para calcular diferentes thresholds
# Para um caso real, você substituiria 'y_true' e 'y_probs' pelas saídas do modelo MLP no Test Loader.
np.random.seed(42)
y_true = np.random.randint(0, 2, 1000)
y_probs = np.random.rand(1000)

cost_fp = 50   # Custo de um Falso Positivo (ex: dar $50 de brinde para quem não cancelaria)
cost_fn = 200  # Custo de um Falso Negativo (ex: perder um CLTV de $200 de um cliente)

thresholds = np.linspace(0.1, 0.9, 9)
costs = []

for t in thresholds:
    y_pred = (y_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = (fp * cost_fp) + (fn * cost_fn)
    costs.append(total_cost)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, costs, marker='o', linestyle='-', color='purple')
plt.title('Trade-off de Custos por Threshold de Decisão (MLP)')
plt.xlabel('Threshold (Limiar de Probabilidade de Churn)')
plt.ylabel('Custo Financeiro Total (Simulado)')
plt.grid(True)
plt.show()

best_threshold = thresholds[np.argmin(costs)]
print(f"-> O threshold ideal para minimizar custos operacionais seria: {best_threshold:.2f}")